# 66 — Documentary Adjudication Patch

Patches the existing site adjudication with documentary evidence strength.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "66_documentary_adjudication_patch"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

adjud, _ = safe_read_csv(OUTPUTS / "56_site_adjudication" / "site_adjudication.csv")
docs, _ = safe_read_csv(OUTPUTS / "65_documentary_evidence_fusion" / "documentary_evidence_site_year.csv")

if adjud.empty:
    raise FileNotFoundError("Need outputs/56_site_adjudication/site_adjudication.csv")
if docs.empty:
    raise FileNotFoundError("Need outputs/65_documentary_evidence_fusion/documentary_evidence_site_year.csv")

docs_max = docs.groupby("site_id", as_index=False).agg(
    documentary_evidence_score=("documentary_evidence_score", "max"),
    documentary_exceedance_rows=("documentary_exceedance_rows", "max"),
    documentary_noncompliance_rows=("documentary_noncompliance_rows", "max"),
    documentary_report_count=("documentary_report_count", "max"),
)

work = adjud.copy()
key_col = "site_id" if "site_id" in work.columns else ("site_name" if "site_name" in work.columns else None)
if key_col != "site_id":
    work["site_id"] = work[key_col].map(slugify)

patched = work.merge(docs_max, on="site_id", how="left")
for c in ["documentary_evidence_score","documentary_exceedance_rows","documentary_noncompliance_rows","documentary_report_count"]:
    if c not in patched.columns:
        patched[c] = 0
    patched[c] = pd.to_numeric(patched[c], errors="coerce").fillna(0)

patched["integrity_score_patched"] = (patched.get("integrity_score", 0).astype(float) * 0.7 + np.clip(patched["documentary_report_count"] / 5, 0, 1) * 0.3).round(3)
patched["signal_score_patched"] = (patched.get("signal_score", 0).astype(float) + patched["documentary_evidence_score"]).round(3)

def patched_verdict(row):
    if row["documentary_exceedance_rows"] > 0 or row["documentary_noncompliance_rows"] > 0:
        return "priority_review"
    if row["documentary_evidence_score"] >= 1.0 and row.get("verdict", "") != "not_ready":
        return "documentary_supported_review"
    return row.get("verdict", "not_ready")

patched["verdict_patched"] = patched.apply(patched_verdict, axis=1)
patched.to_csv(out / "site_adjudication_patched.csv", index=False)

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
add_artifact(manifest, out / "site_adjudication_patched.csv")
write_json(out / "manifest.json", manifest)

display(patched.head(20))